# 📊 Gerador de Painel de Cobrança via WhatsApp Web

Este notebook processa uma planilha de inadimplentes (`.xls` ou `.xlsx`) e gera uma página **HTML autônoma** com:
* Links diretos para o **WhatsApp Web** com mensagem personalizada (singular/plural) e cálculo automático de valores;
* **Controle de contatos** (flag `Contatado?`) com cards dinâmicos de acompanhamento em quantidade, percentual e valor em R$;
* **Campo de observação** persistente e opção para salvar uma cópia do HTML com as anotações embutidas;
* **Exportação limpa para Excel (`.xlsx`)** sem a coluna de links.

---

### ⚙️ Passo 1: Configuração do Atendimento
Preencha os campos abaixo (ou deixe em branco para usar os valores padrão) e execute a célula.

In [ ]:
#@title Parâmetros da Cobrança { run: "auto" }
NOME_ATENDENTE = "" #@param {type:"string"}
NOME_EMPRESA = "" #@param {type:"string"}

atendente_final = NOME_ATENDENTE.strip() or "Nome do atendente"
empresa_final = NOME_EMPRESA.strip() or "Nome da empresa"

print(f"Configurado -> Atendente: '{atendente_final}' | Empresa: '{empresa_final}'")

### 📤 Passo 2: Enviar Planilha e Gerar Painel
Execute a célula abaixo. 
* **No Google Colab:** aparecerá um botão **Escolher arquivos** para selecionar sua planilha e o download do HTML ocorrerá automaticamente ao final.
* **No Jupyter Local / VS Code:** informe o caminho do arquivo quando solicitado.

In [ ]:
import urllib.request
from pathlib import Path

SCRIPT_LOCAL = "gerar_inadimplentes_whatsapp_colab.py"
URL_RAW_SCRIPT = "https://raw.githubusercontent.com/amisadaimodesto/gerador-cobranca-whatsapp/main/gerar_inadimplentes_whatsapp_colab.py"

if not Path(SCRIPT_LOCAL).exists():
    print("Baixando motor de processamento do GitHub...")
    urllib.request.urlretrieve(URL_RAW_SCRIPT, SCRIPT_LOCAL)

import gerar_inadimplentes_whatsapp_colab as gerador

# Garante valores padrão caso a célula do Passo 1 não tenha sido executada antes
_atendente = globals().get("atendente_final", "Nome do atendente")
_empresa = globals().get("empresa_final", "Nome da empresa")

if gerador.esta_no_google_colab():
    caminho_planilha = gerador.selecionar_arquivo_colab()
else:
    caminho_planilha = input("Informe o caminho completo da planilha (.xls ou .xlsx): ").strip().strip('"').strip("'")

linhas, nome_aba = gerador.ler_planilha(caminho_planilha)
registros = gerador.extrair_registros(linhas)

if not registros:
    raise RuntimeError("Não foi possível localizar os blocos de Inadimplentes na planilha.")

conteudo_html = gerador.gerar_html(
    registros,
    nome_aba,
    nome_atendente=_atendente,
    nome_empresa=_empresa,
)

arquivo_saida = gerador.salvar_html(conteudo_html, gerador.NOME_ARQUIVO_SAIDA)

print(f"\n✅ Concluído! {len(registros)} registro(s) processado(s).")
print(f"📄 Arquivo gerado em: {arquivo_saida}")

gerador.tentar_baixar_no_colab(arquivo_saida)